### 1. Leer el diccionario

In [2]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

# Ajusta estas rutas según tu repo
PATH_CSV = Path.cwd().parent / "data" / "raw" / "Enaho01-2025-100.csv"
PATH_DICT = Path.cwd().parent / "docs" / "diccionario_reducido_datamart_enaho2025.xlsx"

In [3]:
dict_vars = pd.read_excel(
    PATH_DICT,
    sheet_name="Diccionario Variables",
    header=2
)

dict_vars = dict_vars.dropna(subset=["Variable original"]).copy()

dict_vars.head()

,Sección / Dimensión,Variable original,Nombre BI sugerido,Significado,Uso,Origen / Regla
0,Identificación / Tiempo / Geografía,AÑO,anio,Año de la encuesta,Dimensión tiempo,ENAHO Módulo 100
1,Identificación / Tiempo / Geografía,MES,mes,Mes de ejecución de la encuesta,Dimensión tiempo,ENAHO Módulo 100
2,Identificación / Tiempo / Geografía,PERIODO,periodo,Periodo de ejecución de la encuesta,Dimensión tiempo,ENAHO Módulo 100
3,Identificación / Tiempo / Geografía,FECENT,fecha_entrevista,Fecha del resultado final de la encuesta,Dimensión tiempo,ENAHO Módulo 100
4,Identificación / Tiempo / Geografía,CONGLOME,conglomerado,Número de conglomerado,Construcción de hogar_id,ENAHO Módulo 100


In [4]:
variables_datamart = dict_vars["Variable original"].astype(str).str.strip().tolist()

len(variables_datamart), variables_datamart[:10] # una muestra pequeña de las variables que queremos en el datamart

(91,
 ['AÑO',
  'MES',
  'PERIODO',
  'FECENT',
  'CONGLOME',
  'VIVIENDA',
  'HOGAR',
  'UBIGEO',
  'DOMINIO',
  'ESTRATO'])

### 2. Cargar solo las columnas del datamart

In [5]:
# 1. Leer solo los nombres de columnas reales del CSV
columnas_csv = pd.read_csv(
    PATH_CSV,
    sep=";",
    encoding="latin1",
    dtype=str,
    nrows=0
).columns.str.strip().tolist()

columnas_csv_set = set(columnas_csv)

In [6]:
# 2. Expandir variables tipo I1172$XX, I1173$XX, I1174$XX
variables_expandidas = []

for var in variables_datamart:
    var = str(var).strip()
    
    if var.endswith("$XX"):
        prefijo = var.replace("$XX", "$")
        
        columnas_match = [
            col for col in columnas_csv
            if col.startswith(prefijo)
        ]
        
        variables_expandidas.extend(columnas_match)
    else:
        variables_expandidas.append(var)

# Quitar duplicados manteniendo el orden
variables_expandidas = list(dict.fromkeys(variables_expandidas))

len(variables_datamart), len(variables_expandidas)

(91, 136)

In [7]:
faltantes = [
    var for var in variables_expandidas
    if var not in columnas_csv_set
]

faltantes

[]

In [8]:
df_raw = pd.read_csv(
    PATH_CSV,
    sep=";",
    encoding="latin1",
    dtype=str,
    keep_default_na=False,
    usecols=variables_expandidas
)

print(df_raw.shape)
df_raw.head()

(44599, 136)


,AÑO,MES,CONGLOME,VIVIENDA,HOGAR,UBIGEO,DOMINIO,ESTRATO,PERIODO,FECENT,RESULT,P101,P102,P103,P103A,P104,P104A,P105A,P105B,P106,P106A,P106B,P107B1,P107D1,P107B2,P107D2,P107B3,P107D3,P107B4,P107D4,P110,P110A1,P110A,P110A_MODIFICADA,P110C,P110C1,P110C2,P110C3,P110D,P110E,P111A,P1121,P1123,P1124,P1125,P1126,P1127,P112A,P1131,P1132,P1133,P1135,P1136,P1139,P1137,P1138,P113A,P1141,P1142,P1143,...,I1172$01,I1172$02,I1172$04,I1172$05,I1172$06,I1172$07,I1172$08,I1172$09,I1172$10,I1172$11,I1172$12,I1172$13,I1172$14,I1172$17,I1172$15,I1172$16,I1173$01,I1174$01,I1173$02,I1174$02,I1173$04,I1174$04,I1173$05,I1174$05,I1173$06,I1174$06,I1173$07,I1174$07,I1173$08,I1174$08,I1173$09,I1174$09,I1173$10,I1174$10,I1173$11,I1174$11,I1173$12,I1174$12,I1173$13,I1174$13,I1173$14,I1174$14,I1173$15,I1174$15,I1173$16,I1174$16,I1173$17,I1174$17,T111A,NBI1,NBI2,NBI3,NBI4,NBI5,FACTOR07,CODCCPP,NOMCCPP,LONGITUD,LATITUD,ALTITUD
0,2025,01,015009,012,11,010101,4,4,2,20250113,1,1,1,2,1,4,2,2,,780,1,1,2,,2,,2,,2,,1,1,1,1,1,24,,,1,1,1,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,2,0,1,0,...,180,240,324,,,,,,,,1019,,0,,,,0,,0,,0,0,,,,,,,,,,,,,,,0,,,,0,,,,,,,,1,0,0,0,0,0,"87,2856903076172",0001,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301"
1,2025,01,015009,025,11,010101,4,4,2,20250114,1,1,3,5,4,3,3,2,,225,2,,2,,2,,2,,2,,1,1,1,",8",1,24,,,1,1,1,1,0,0,0,0,0,1,0,1,0,0,1,0,1,0,2,0,1,0,...,168,216,959,,,,0,,,,925,,0,,0,,0,,0,,0,0,,,,,,,336,0,,,,,,,0,,,,0,,0,0,,,,,1,0,0,0,0,0,"87,2856903076172",0001,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301"
2,2025,01,015009,036,11,010101,4,4,2,20250113,5,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,...,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,"87,2856903076172",0001,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301"
3,2025,01,015009,049,11,010101,4,4,2,20250125,1,1,1,5,1,3,2,2,,785,1,1,2,,2,,2,,2,,1,1,1,",6",1,24,,,1,1,1,1,0,0,0,0,0,1,1,1,0,0,1,0,0,0,2,0,1,1,...,419,959,515,,,,240,,,,937,293,0,,,,0,,0,,0,0,,,,,,,0,0,,,,,,,0,,0,,0,,,,,,,,1,0,0,0,0,0,"87,2856903076172",0001,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301"
4,2025,01,015009,061,11,010101,4,4,2,20250203,1,1,3,5,5,2,1,2,,180,1,1,2,,2,,2,,2,,1,1,1,"1,2",1,24,,,1,1,1,1,0,0,0,0,0,2,0,1,0,0,0,0,0,0,2,0,1,0,...,0,0,346,,,,,,,,0,,0,,,,155,,215,,0,0,,,,,,,,,,,,,,,106,,,,0,,,,,,,,1,0,0,0,0,0,"87,2856903076172",0001,CIUDAD CHACHAPOYAS,"-77,869230754","-6,234386829","2350,55706301"


### 3. Espacios en blanco

In [9]:
df = df_raw.copy()

for col in df.columns:
    df[col] = df[col].str.strip()

df = df.replace("", pd.NA)

### 4. revisar resultado de encuesta

El diccionario recomienda filtrar RESULT = 1, es decir, entrevistas completas.

In [10]:
df["RESULT"].value_counts(dropna=False).sort_index()

RESULT
1    27260
2     6442
3     1586
4      294
5     3041
7     5976
Name: count, dtype: int64

In [26]:
df_complete = df[df["RESULT"] == "1"].copy()

df_complete.shape

(27260, 136)

### 5. Completar y validar hogar_id

Nuestra granularidad es 1 hogar por fila

In [28]:
df_complete["hogar_id"] = (
    df_complete["AÑO"].astype(str) + "_" +
    df_complete["CONGLOME"].astype(str) + "_" +
    df_complete["VIVIENDA"].astype(str) + "_" +
    df_complete["HOGAR"].astype(str)
)

df_complete[["AÑO", "CONGLOME", "VIVIENDA", "HOGAR", "hogar_id"]].head()

,AÑO,CONGLOME,VIVIENDA,HOGAR,hogar_id
0,2025,015009,012,11,2025_015009_012_11
1,2025,015009,025,11,2025_015009_025_11
3,2025,015009,049,11,2025_015009_049_11
4,2025,015009,061,11,2025_015009_061_11
5,2025,015009,074,11,2025_015009_074_11


In [29]:
duplicados_hogar = df_complete.duplicated(subset=["AÑO", "CONGLOME", "VIVIENDA", "HOGAR"]).sum()

print("Duplicados por llave natural:", duplicados_hogar)
print("Hogares únicos:", df_complete["hogar_id"].nunique())
print("Filas:", len(df_complete))

Duplicados por llave natural: 0
Hogares únicos: 27260
Filas: 27260


### 6. Profiling de nulos

Varios de estos nulos no son errores, sino saltos lógicos del cuestionario. Por ejemplo, los montos de crédito solo aparecen si el hogar recibió crédito.

In [30]:
perfil_nulos = (
    df_complete
    .isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "variable", 0: "nulos"})
)

perfil_nulos["porcentaje_nulos"] = perfil_nulos["nulos"] / len(df_complete) * 100

perfil_nulos = perfil_nulos.sort_values("porcentaje_nulos", ascending=False)

perfil_nulos.head(10)

,variable,nulos,porcentaje_nulos
111,I1174$11,27260,100.000000
93,I1174$01,27260,100.000000
99,I1174$05,27260,100.000000
95,I1174$02,27260,100.000000
123,I1174$17,27260,100.000000
113,I1174$12,27260,100.000000
115,I1174$13,27260,100.000000
117,I1174$14,27260,100.000000
23,P107D1,27216,99.838591
70,D107D1,27216,99.838591


In [31]:
# REPORTES DE PROFILING

OUTPUT_DIR = Path("outputs/profiling")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

perfil_nulos.to_csv(
    OUTPUT_DIR / "perfil_nulos_datamart.csv",
    index=False,
    encoding="utf-8-sig"
)

In [32]:
# PROFILING DE CATEGÓRICAS CLAVE

'''
En la variable P113A aparece el código 9 en 400 hogares.

También puedo ver que no aparece el código 8 en esos hogares completos, pero eso no significa 
que el código 8 no exista como categoría posible; solo significa que no apareció en ese subconjunto filtrado.
'''

categoricas_clave = [
    "DOMINIO", "ESTRATO", "PERIODO",
    "P101", "P102", "P103", "P103A",
    "P105A",
    "P110", "P110A1", "P110A", "P110C",
    "P111A", "T111A",
    "P113A",
    "P1141", "P1142", "P1143", "P1144", "P1145", "P1146",
    "NBI1", "NBI2", "NBI3", "NBI4", "NBI5"
]

for col in categoricas_clave:
    print("\n" + "="*60)
    #print(col)
    print(df_complete[col].value_counts(dropna=False).sort_index())


DOMINIO
1    4234
2    2523
3    1560
4    1902
5    5212
6    3569
7    6770
8    1490
Name: count, dtype: int64

ESTRATO
1    2978
2    5078
3    2257
4    2323
5    3922
6    1826
7    6265
8    2611
Name: count, dtype: int64

PERIODO
1    6387
2    7420
3    7588
4    5347
5     518
Name: count, dtype: int64

P101
1       25023
2         747
3         154
4         836
5         204
6          10
8           1
<NA>      285
Name: count, dtype: int64

P102
1       12618
2          91
3        7048
4        2208
5         350
6         204
7        3249
8         599
9         608
<NA>      285
Name: count, dtype: int64

P103
1         328
2         948
3        3137
4        2663
5       11708
6        8154
7          37
<NA>      285
Name: count, dtype: int64

P103A
1        7755
2         293
3        2003
4       15718
5         695
6          90
7         365
8          56
<NA>      285
Name: count, dtype: int64

P105A
1     2210
2    19533
3     1215
4       70
5       75
6   

### 7. Formateo numérico

In [33]:
def to_number(series):
    return (
        series
        .astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False)
        .replace("", pd.NA)
        .pipe(pd.to_numeric, errors="coerce")
    )

In [34]:
columnas_numericas = [
    "FACTOR07",
    "P104", "P104A",
    "P105B", "P106", "I105B", "I106",
    "P110A_MODIFICADA", "P110C1", "P110C2", "P110C3",
    "P107D1", "P107D2", "P107D3", "P107D4",
    "D107D1", "D107D2", "D107D3", "D107D4",
    "LONGITUD", "LATITUD", "ALTITUD"
]

for col in columnas_numericas:
    if col in df_complete.columns:
        df_complete[col + "_num"] = to_number(df_complete[col])

### 8. Profiling de medidas numéricas

In [35]:
perfil_numerico = df_complete[[col + "_num" for col in columnas_numericas if col in df_complete.columns]].describe().T

perfil_numerico

,count,mean,std,min,25%,50%,75%,max
FACTOR07_num,27260.0,276.373282,259.278794,1.66457,115.982826,214.708984,348.816071,1845.740967
P104_num,26975.0,3.204337,1.613409,1.0,2.0,3.0,4.0,15.0
P104A_num,26975.0,1.868063,1.211972,0.0,1.0,2.0,3.0,13.0
P105B_num,2280.0,845.030702,6594.370973,10.0,150.0,300.0,510.0,99999.0
P106_num,25050.0,2699.003553,15338.308761,5.0,65.0,160.0,390.0,99999.0
I105B_num,2280.0,4907.683991,4753.7827,121.0,1812.0,3553.5,6087.0,47497.0
I106_num,25050.0,3391.364032,4675.659052,59.0,777.0,1884.5,4379.0,202142.0
P110A_MODIFICADA_num,27260.0,0.233122,0.427779,0.0,0.0,0.0,0.4,3.4
P110C1_num,20197.0,18.128435,8.580571,1.0,10.0,24.0,24.0,24.0
P110C2_num,3176.0,2.923804,1.210656,1.0,2.0,3.0,4.0,6.0


In [36]:
perfil_numerico.to_csv(
    OUTPUT_DIR / "perfil_numerico_datamart.csv",
    encoding="utf-8-sig"
)

### 9. Primeras reglas de calidad

In [37]:
checks = {}

checks["hogar_id_duplicado"] = df_complete["hogar_id"].duplicated().sum()

checks["habitaciones_total_menor_dormir"] = (
    df_complete["P104_num"] < df_complete["P104A_num"]
).sum()

checks["factor_expansion_nulo"] = df_complete["FACTOR07_num"].isna().sum()

checks["latitud_nula"] = df_complete["LATITUD_num"].isna().sum()
checks["longitud_nula"] = df_complete["LONGITUD_num"].isna().sum()

checks["agua_todos_dias_sin_horas"] = (
    (df_complete["P110C"] == "1") &
    (df_complete["P110C1_num"].isna())
).sum()

checks["agua_no_todos_dias_sin_dias"] = (
    (df_complete["P110C"] == "2") &
    (df_complete["P110C2_num"].isna())
).sum()

checks["agua_no_todos_dias_sin_horas"] = (
    (df_complete["P110C"] == "2") &
    (df_complete["P110C3_num"].isna())
).sum()

checks

{'hogar_id_duplicado': np.int64(0),
 'habitaciones_total_menor_dormir': np.int64(0),
 'factor_expansion_nulo': np.int64(0),
 'latitud_nula': np.int64(0),
 'longitud_nula': np.int64(0),
 'agua_todos_dias_sin_horas': np.int64(0),
 'agua_no_todos_dias_sin_dias': np.int64(0),
 'agua_no_todos_dias_sin_horas': np.int64(0)}

# Crear tabla intermedia

In [38]:
df_complete.shape

(27260, 159)

In [39]:
df_profiled = df_complete.copy()

# Crear hogar_id si todavía no existe
if "hogar_id" not in df_profiled.columns:
    df_profiled["hogar_id"] = (
        df_profiled["AÑO"].astype("string").str.strip() + "_" +
        df_profiled["CONGLOME"].astype("string").str.strip() + "_" +
        df_profiled["VIVIENDA"].astype("string").str.strip() + "_" +
        df_profiled["HOGAR"].astype("string").str.strip()
    )

# Validación básica
assert df_profiled["RESULT"].eq("1").all(), "Hay registros que no tienen RESULT = 1"
assert df_profiled["hogar_id"].duplicated().sum() == 0, "Hay hogares duplicados"

print("Filas de tabla intermedia:", len(df_profiled))
print("Columnas de tabla intermedia:", len(df_profiled.columns))

Filas de tabla intermedia: 27260
Columnas de tabla intermedia: 159


In [41]:
INTERIM_DIR = Path("../data/interim")
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

PATH_PROFILED = INTERIM_DIR / "enaho100_profiled.parquet"
PATH_PROFILED_CSV = INTERIM_DIR / "enaho100_profiled.csv"

In [44]:
try:
    df_profiled.to_parquet(
        PATH_PROFILED,
        index=False
    )
except Exception as e:
    print("to_parquet falló:", type(e).__name__, e)
    df_profiled.to_csv(
        PATH_PROFILED_CSV,
        index=False,
        encoding="utf-8-sig",
    )

print("Base intermedia guardada en:")
print(PATH_PROFILED)
print(PATH_PROFILED_CSV)

to_parquet falló: ArrowKeyError A type extension with name pandas.period already defined
Base intermedia guardada en:
..\data\interim\enaho100_profiled.parquet
..\data\interim\enaho100_profiled.csv
